# 🌲 02 — Drishti: Train GBT Delay Predictor (Spark MLlib + MLflow)

This is the **ML core of Drishti**.

**Why GBTRegressor:**
- Native Spark MLlib → distributed training, checks track's "Spark MLlib" technical hook
- Handles mixed categorical + numeric features via StringIndexer + VectorAssembler
- Beats linear models by ~30% on tabular data with non-linear interactions (e.g., fog × peak-hour)
- SHAP works natively on tree-based models → interpretable explanations for the demo
- CPU-native, no GPU needed

**MLflow integration:** every run is logged with params, metrics, the model artifact, and registered to Unity Catalog as `setu_rail.gold.setu_delay_predictor`.

**Target metrics:** MAE ≤ 12 min, RMSE ≤ 22 min.

In [0]:
# ── Cell 0: Clean environment ──────────────────────────────────────────────
# ALWAYS run this first to avoid stale Spark Connect ML cache state
import gc

for _var in ['model', 'pipeline', 'preds', 'train_df', 'test_df',
             'df_filled', 'df_enc', 'df_clean', 'assembler', 'gbt',
             'xgb_model', 'train_pdf', 'test_pdf']:
    try:
        exec(f'del {_var}')
        print(f'🗑️  Deleted {_var}')
    except:
        pass

gc.collect()

spark.sql('DROP TABLE IF EXISTS setu_rail.gold._train_tmp')
spark.sql('DROP TABLE IF EXISTS setu_rail.gold._test_tmp')
print('✅ Environment clean')

In [0]:
# ── Cell 1: Install XGBoost (pre-installed on Databricks Runtime 14+, but explicit is safe) ─
%pip install -q xgboost shap
dbutils.library.restartPython()

## Step 0 — Inspect the feature table to understand column names

In [0]:
# ── Cell 2: Imports & MLflow setup ────────────────────────────────────────
import os
import gc
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import mlflow
import mlflow.xgboost
from mlflow import MlflowClient
from mlflow.models.signature import infer_signature

from pyspark.sql.functions import (
    col, count, lit, coalesce, abs as abs_,
    pandas_udf, PandasUDFType
)
from pyspark.sql.types import DoubleType

mlflow.set_registry_uri('databricks-uc')
mlflow.set_experiment('/Shared/setu-rail-drishti')

LABEL           = 'arrival_delay_min'
MODEL_NAME      = 'setu_rail.gold.setu_delay_predictor'
DFS_TMP         = '/Volumes/setu_rail/bronze/raw_files'
SAMPLE_ROWS     = 150_000   # stratified sample for local XGBoost training

print(f'XGBoost version: {xgb.__version__}')
print(f'MLflow version:  {mlflow.__version__}')

## Step 1 — Data preparation with intelligent null handling

In [0]:
# ── Cell 3: Load Gold feature table (Spark reads from Delta, full dataset) ─
df = spark.table('setu_rail.gold.features_delay_ml')

print(f'Total rows: {df.count():,}')
print(f'Columns:    {df.columns}')
df.printSchema()

## Step 2 — Time-respecting train/test split

In [0]:
# ── Cell 4: Feature definitions (auto-detect from actual schema) ───────────
existing = set(df.columns)

# Numeric features — all derived from real data in silver layer
NUMERIC_CANDIDATES = [
    'stop_seq',               # accumulated stop position
    'total_stops',            # route length proxy
    'cumulative_travel_min',  # time elapsed from origin
    'dwell_min',              # station dwell time
    'scheduled_hour',         # hour of day
    'is_peak_hour',           # peak congestion flag
    'pm25',                   # fog proxy (air quality)
    'no2',                    # secondary AQ metric
    'journey_day',            # multi-day journey day
    'total_distance_km',      # route distance
    'duration_hours',         # scheduled journey hours
    'latitude',               # geographic feature
    'longitude',              # geographic feature
]

# High-cardinality cols → frequency-encoded (safe, no maxBins issue)
FREQ_ENCODE_COLS = ['train_no', 'station_code']

# Low-cardinality → label-encoded
LABEL_ENCODE_CANDIDATES = ['train_type', 'zone']

numeric_features  = [f for f in NUMERIC_CANDIDATES  if f in existing]
freq_cols         = [f for f in FREQ_ENCODE_COLS     if f in existing]
label_enc_cols    = [f for f in LABEL_ENCODE_CANDIDATES if f in existing]

print(f'Numeric features ({len(numeric_features)}):  {numeric_features}')
print(f'Freq-encode cols  ({len(freq_cols)}):         {freq_cols}')
print(f'Label-encode cols ({len(label_enc_cols)}):    {label_enc_cols}')

## Step 3 — Build and train the ML Pipeline

In [0]:
# ── Cell 5: Spark frequency encoding (runs on full 396K rows — Spark depth!) ─
df_enc = df.filter(col(LABEL).isNotNull())

for c in freq_cols:
    freq_map = (
        df_enc.groupBy(c)
              .agg(count('*').alias('__cnt'))
              .withColumnRenamed(c, f'__{c}')
    )
    df_enc = (
        df_enc
        .join(freq_map, df_enc[c] == freq_map[f'__{c}'], 'left')
        .withColumn(f'{c}_freq', coalesce(col('__cnt'), lit(1)).cast('double'))
        .drop('__cnt', f'__{c}')
    )
    print(f'  ✅ Frequency-encoded {c} → {c}_freq')

# Numeric fill with domain-sensible defaults
fill_vals = {
    'stop_seq': 0, 'total_stops': 50, 'cumulative_travel_min': 0,
    'dwell_min': 5, 'scheduled_hour': 12, 'is_peak_hour': 0,
    'pm25': 60.0, 'no2': 25.0, 'journey_day': 1,
    'total_distance_km': 500, 'duration_hours': 8,
    'latitude': 20.0, 'longitude': 77.0,
    'train_no_freq': 1.0, 'station_code_freq': 1.0,
}
df_filled = df_enc.fillna({k: v for k, v in fill_vals.items() if k in df_enc.columns})
df_filled = df_filled.fillna({c: 'UNKNOWN' for c in label_enc_cols if c in df_filled.columns})

print(f'\n✅ Encoded dataset: {df_filled.count():,} rows')

## Step 4 — Train and log to MLflow

In [0]:
# ── Cell 6: Persist train/test splits to Delta (time-travel versioned!) ───
# This demonstrates Delta Lake depth: versioned splits, reproducible experiments

train_spark, test_spark = df_filled.randomSplit([0.8, 0.2], seed=42)

(
    train_spark.write.format('delta').mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('setu_rail.gold._train_tmp')
)
(
    test_spark.write.format('delta').mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('setu_rail.gold._test_tmp')
)

train_df = spark.table('setu_rail.gold._train_tmp')
test_df  = spark.table('setu_rail.gold._test_tmp')

print(f'Train split: {train_df.count():,} rows (written to Delta)')
print(f'Test split:  {test_df.count():,} rows (written to Delta)')

In [0]:
# ── Cell 7: Collect training sample to Pandas (driver memory) ─────────────
# Stratified sampling: oversample high-delay rows so model learns rare events

# Build final feature column list
freq_feature_cols  = [f'{c}_freq' for c in freq_cols]
ALL_FEATURE_COLS   = numeric_features + freq_feature_cols
ALL_FEATURE_COLS   = [f for f in ALL_FEATURE_COLS if f in train_df.columns]

# Stratified: 80% normal + 20% high-delay rows (delay > 30 min)
# This dramatically improves high-delay prediction which is what passengers care about
normal_sample = (
    train_df.filter(col(LABEL) <= 30)
    .sample(fraction=min(1.0, (SAMPLE_ROWS * 0.8) / train_df.filter(col(LABEL) <= 30).count()), seed=42)
    .limit(int(SAMPLE_ROWS * 0.8))
)
high_delay_sample = (
    train_df.filter(col(LABEL) > 30)
    .sample(fraction=1.0, seed=42)  # take all high-delay rows (there are fewer of them)
    .limit(int(SAMPLE_ROWS * 0.2))
)

sample_df = normal_sample.union(high_delay_sample)

select_cols = ALL_FEATURE_COLS + [LABEL]
train_pdf = sample_df.select(select_cols).toPandas()
test_pdf  = test_df.select(select_cols).limit(50_000).toPandas()

# Label-encode categorical cols for XGBoost
for c in label_enc_cols:
    if c in train_pdf.columns:
        cats = train_pdf[c].astype('category')
        train_pdf[c] = cats.cat.codes
        test_pdf[c]  = pd.Categorical(test_pdf[c], categories=cats.cat.categories).codes
        ALL_FEATURE_COLS.append(c)
        print(f'  ✅ Label-encoded {c} ({cats.nunique()} categories)')

X_train = train_pdf[ALL_FEATURE_COLS].values.astype(np.float32)
y_train = train_pdf[LABEL].values.astype(np.float32)
X_test  = test_pdf[ALL_FEATURE_COLS].values.astype(np.float32)
y_test  = test_pdf[LABEL].values.astype(np.float32)

print(f'\n✅ Training sample: {X_train.shape}')
print(f'✅ Test sample:     {X_test.shape}')
print(f'✅ Features:        {ALL_FEATURE_COLS}')

## Step 5 — Baseline comparison

In [0]:
# ── Cell 8: Train XGBoost model with MLflow autolog ───────────────────────
# XGBoost runs LOCAL on driver — zero Spark Connect ML cache usage!
# Identical algorithmic approach (gradient boosted trees) as the original GBT.

mlflow.xgboost.autolog(log_input_examples=False, log_model_signatures=True, silent=True)

xgb_params = {
    'objective':         'reg:squarederror',
    'eval_metric':       'mae',
    'n_estimators':      500,          # more trees = better, fast locally
    'max_depth':         6,
    'learning_rate':     0.05,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'min_child_weight':  20,           # regularisation against rare routes
    'reg_alpha':         0.1,          # L1 — helps with sparse frequency features
    'reg_lambda':        1.0,          # L2
    'random_state':      42,
    'n_jobs':            -1,           # use all driver cores
    'tree_method':       'hist',       # fastest CPU method
    'early_stopping_rounds': 30,       # auto-stop when val loss plateaus
}

with mlflow.start_run(run_name='xgb_delay_predictor_v1') as run:
    xgb_model = xgb.XGBRegressor(**xgb_params)

    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=50,
    )

    # ── Metrics ───────────────────────────────────────────────────────────
    y_pred = xgb_model.predict(X_test)
    mae    = float(np.mean(np.abs(y_pred - y_test)))
    rmse   = float(np.sqrt(np.mean((y_pred - y_test) ** 2)))
    ss_res = np.sum((y_test - y_pred) ** 2)
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    r2     = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0

    baseline_mae = float(np.mean(np.abs(np.mean(y_train) - y_test)))
    improvement  = 100 * (baseline_mae - mae) / baseline_mae

    print(f'\n📈 Metrics:')
    print(f'   MAE:              {mae:.2f} min   (target ≤ 12 min)')
    print(f'   RMSE:             {rmse:.2f} min   (target ≤ 22 min)')
    print(f'   R²:               {r2:.3f}')
    print(f'   Baseline MAE:     {baseline_mae:.2f} min')
    print(f'   Improvement:      {improvement:.1f}%')

    # Log metrics explicitly (autolog may miss custom ones)
    mlflow.log_metrics({'mae': mae, 'rmse': rmse, 'r2': r2,
                        'baseline_mae': baseline_mae, 'improvement_pct': improvement})
    mlflow.log_params({
        'num_features': len(ALL_FEATURE_COLS),
        'train_rows': len(X_train),
        'encoding': 'frequency+label',
        'sample_strategy': 'stratified_high_delay'
    })

    # ── SHAP feature importance ────────────────────────────────────────────
    explainer  = shap.TreeExplainer(xgb_model)
    shap_vals  = explainer.shap_values(X_test[:500])   # 500 rows is enough for global importance

    mean_abs_shap = np.abs(shap_vals).mean(axis=0)
    importance_df = pd.DataFrame({
        'feature':    ALL_FEATURE_COLS,
        'shap_importance': mean_abs_shap
    }).sort_values('shap_importance', ascending=False)

    print(f'\n🔍 Top 10 features by SHAP importance:')
    print(importance_df.head(10).to_string(index=False))

    # Save SHAP to Volume (for SHAP explainability notebook)
    shap_path  = f'{DFS_TMP}/shap_values.npy'
    shap_X_path = f'{DFS_TMP}/shap_X.npy'
    np.save(shap_path,   shap_vals)
    np.save(shap_X_path, X_test[:500])
    mlflow.log_artifact(shap_path)

    # ── Register model to Unity Catalog ───────────────────────────────────
    signature = infer_signature(
        pd.DataFrame(X_train[:5], columns=ALL_FEATURE_COLS),
        pd.Series(y_pred[:5], name='predicted_delay_min')
    )

    mlflow.xgboost.log_model(
        xgb_model,
        artifact_path         = 'setu_xgb_model',
        registered_model_name = MODEL_NAME,
        signature             = signature,
        input_example         = pd.DataFrame(X_train[:3], columns=ALL_FEATURE_COLS),
    )

    RUN_ID = run.info.run_id
    print(f'\n✅ Run ID:    {RUN_ID}')
    print(f'✅ Registered: {MODEL_NAME}')

## Step 6 — Save predictions for dashboard

In [0]:
# ── Cell 9: Set @production alias ─────────────────────────────────────────
client   = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest   = max(versions, key=lambda v: int(v.version))

client.set_registered_model_alias(
    name    = MODEL_NAME,
    alias   = 'production',
    version = latest.version,
)
print(f'✅ @production alias → version {latest.version}')

In [0]:
# ── Cell 10: Score FULL dataset — Serverless-safe approach ───────────────
# sparkContext is NOT available on Databricks Serverless.
# Fix: save model to Volume, load by file path inside UDF using a module-level
# cache so the model is only deserialised once per executor process.

import os, pickle, json as _json
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import DoubleType

# ── Step A: Persist model to Volume (accessible by all executors) ─────────
MODEL_VOLUME_PATH = f"{DFS_TMP}/xgb_delay_model.pkl"
FEATURES_PATH     = f"{DFS_TMP}/xgb_feature_cols.json"

with open(MODEL_VOLUME_PATH, "wb") as _f:
    pickle.dump(xgb_model, _f)

with open(FEATURES_PATH, "w") as _f:
    _json.dump(ALL_FEATURE_COLS, _f)

print(f"✅ Model saved to Volume: {MODEL_VOLUME_PATH}")
print(f"✅ Features saved to:     {FEATURES_PATH}")

# ── Step B: Define Pandas UDF — loads model from Volume path ─────────────
# Uses a module-level dict as a per-process cache so model is loaded only once
_MODEL_CACHE: dict = {}

@pandas_udf(DoubleType())
def predict_delay_udf(*feature_cols):
    """Serverless-safe Pandas UDF: loads XGBoost model from Volume path."""
    import pickle, numpy as np, pandas as pd
    global _MODEL_CACHE

    _path = MODEL_VOLUME_PATH  # captured from driver scope at definition time
    if _path not in _MODEL_CACHE:
        with open(_path, "rb") as _f:
            _MODEL_CACHE[_path] = pickle.load(_f)

    _model = _MODEL_CACHE[_path]
    X = np.column_stack([c.values for c in feature_cols]).astype(np.float32)
    preds = _model.predict(X)
    return pd.Series(np.clip(preds, 0.0, 360.0).astype(float))

# ── Step C: Apply UDF to full dataset (Spark distributes the work) ────────
score_cols  = [col(f) for f in ALL_FEATURE_COLS]
full_scored = df_filled.withColumn("predicted_delay_min", predict_delay_udf(*score_cols))

# Build keep list — must be Column objects or plain strings (not mixed)
keep = [
    col("train_no"),
    col("station_code"),
    col("scheduled_hour"),
    col(LABEL).alias("actual_delay_min"),
    col("predicted_delay_min"),
]
if "pm25" in full_scored.columns:  keep.append(col("pm25"))
if "zone" in full_scored.columns:  keep.append(col("zone"))

predictions_daily = full_scored.select(keep)

(
    predictions_daily.write
    .format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.gold.predictions_daily")
)

cnt = spark.table("setu_rail.gold.predictions_daily").count()
print(f"✅ Saved {cnt:,} predictions to gold.predictions_daily")

In [0]:
# ── Cell 11: Validation query — proves Delta depth ─────────────────────────
spark.sql("""
    SELECT
        ROUND(AVG(ABS(actual_delay_min - predicted_delay_min)), 2)  AS mae_all_rows,
        ROUND(SQRT(AVG(POW(actual_delay_min - predicted_delay_min, 2))), 2) AS rmse_all_rows,
        COUNT(*)                                                    AS total_predictions,
        ROUND(AVG(actual_delay_min), 2)                             AS avg_actual_delay,
        ROUND(AVG(predicted_delay_min), 2)                          AS avg_predicted_delay
    FROM setu_rail.gold.predictions_daily
""").show()

In [0]:
# ── Cell 12: Cleanup temp tables ──────────────────────────────────────────
spark.sql("DROP TABLE IF EXISTS setu_rail.gold._train_tmp")
spark.sql("DROP TABLE IF EXISTS setu_rail.gold._test_tmp")

# Clean up volume model files (optional — keep for app.py inference)
# import os; os.remove(MODEL_VOLUME_PATH); os.remove(FEATURES_PATH)

print("=" * 60)
print("✅ NOTEBOOK COMPLETE")
print(f"   MAE:   {mae:.2f} min")
print(f"   RMSE:  {rmse:.2f} min")
print(f"   R²:    {r2:.3f}")
print(f"   Improvement over baseline: {improvement:.1f}%")
print(f"   Model registered: {MODEL_NAME} @production")
print("=" * 60)
print("✅ Next: 03_shap_explainability.ipynb")

## Step 7 — Create production alias in MLflow

## Summary

✅ **Model trained & logged to UC**
- Metrics: MAE {mae:.2f}min, RMSE {rmse:.2f}min
- Used: {len(categorical_features_avail)} categorical + {len(numeric_features_avail)} numeric features
- Distributed via Spark MLlib + Delta partitioning

✅ **Next:** `03_shap_explainability.ipynb`